# Загрузка и подготовка данных dataset_rework

**Шаг 2 плана.** Загрузка dataset_rework, подготовка сессий, переименование signal_barrier → rd_regime, добавление rd_regime_transition и фичей. Сохранение для следующих ноутбуков.

## 1. Импорты и настройка путей

In [ ]:
import sys
import os

_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) in ('01_data_prep','02_targets','03_features','04_models','05_experiments') else os.getcwd()
if _root not in sys.path:
    sys.path.insert(0, _root)

import pandas as pd
from src.data.dataset_rework_loader import load_dataset_rework
from src.data.data_prep_dataset_rework import prepare_for_training
from src.features.feature_pipeline import add_features, get_feature_columns

## 2. Загрузка dataset_rework и подготовка сессий

In [ ]:
data_dir = os.path.join(_root, 'dataset_rework')
df_raw = load_dataset_rework(data_dir=data_dir, verbose=True)
df = prepare_for_training(df=df_raw, verbose=True)
df.head()

## 3. Переименование signal_barrier → rd_regime, rd_regime_transition

In [ ]:
df = df.rename(columns={'signal_barrier': 'rd_regime'})
df['rd_regime_prev'] = df.groupby('session_key')['rd_regime'].shift(1)
df['rd_regime_transition'] = ((df['rd_regime'] != df['rd_regime_prev']) & df['rd_regime_prev'].notna()).astype(int)
df = df.drop(columns=['rd_regime_prev'], errors='ignore')
print('rd_regime:', df['rd_regime'].value_counts().to_dict())
print('rd_regime_transition (доля переходов):', f"{df['rd_regime_transition'].mean():.4f}")

## 4. Добавление фичей

In [ ]:
df, enc = add_features(df)
feat_cols = get_feature_columns(include_symbol=True)
print(f'Фичей: {len(feat_cols)}')
df[feat_cols[:5] + ['rd_regime']].head()

## 5. Сохранение prepared данных

In [ ]:
out_dir = os.path.join(_root, 'outputs')
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, 'prepared_with_rd_regime.parquet')
df_save = df.copy()
df_save['symbol'] = df_save['symbol'].astype(str)
df_save.to_parquet(out_path, index=False, compression='snappy')
print(f'Сохранено: {out_path} ({len(df):,} строк)')